# ============================================
# AULA 3 — Embeddings e Busca Semântica
# Residência em IA Generativa — PUC Rio 2026
# ============================================

# Instala as bibliotecas necessárias

Lembrete ( carregue os 3 arquivos da auala 2 .md

In [23]:




!pip install sentence-transformers numpy -q

print("✅ Bibliotecas instaladas!")

✅ Bibliotecas instaladas!


#Importações

In [24]:
import numpy as np
import os
import json
from pathlib import Path
from sentence_transformers import SentenceTransformer

print("✅ Bibliotecas importadas!")


✅ Bibliotecas importadas!


#Funções de distância

Distância Euclidiana

In [25]:
def distancia_euclidiana(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    Calcula a Distância Euclidiana entre dois vetores.

    Fórmula: d = sqrt( (v1_1 - v2_1)² + (v1_2 - v2_2)² + ... )

    Analogia: distância em linha reta com uma régua 📏

    Parâmetros:
        vec1, vec2: arrays NumPy de mesma dimensão

    Retorna:
        float: distância (0 = idênticos, quanto maior = mais diferentes)
    """
    # numpy.linalg.norm já calcula a raiz da soma dos quadrados
    return float(np.linalg.norm(vec1 - vec2))


# ============================================
# TESTE RÁPIDO: Entenda a função!
# ============================================
a = np.array([0, 0])
b = np.array([3, 4])

# Visualmente: triângulo retângulo 3-4-5
# d = sqrt(3² + 4²) = sqrt(9 + 16) = sqrt(25) = 5
print(f"Distância entre {a} e {b}: {distancia_euclidiana(a, b)}")
print(f"Esperado: 5.0 ✅" if abs(distancia_euclidiana(a, b) - 5.0) < 0.01 else "❌")


Distância entre [0 0] e [3 4]: 5.0
Esperado: 5.0 ✅


Similaridade cosseno

In [26]:
def similaridade_cosseno(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    Calcula a Similaridade de Cosseno entre dois vetores.

    Fórmula: cos(θ) = (A·B) / (||A|| × ||B||)

    Onde:
        A·B = produto escalar (soma de cada coordenada multiplicada)
        ||A|| = norma/comprimento do vetor

    Retorna:
        float entre -1 e 1
        - 1.0  = mesma direção (idênticos)
        - 0.0  = direções perpendiculares (sem relação)
        - -1.0 = direções opostas

    Se um vetor for zero, retorna 0.0 (evita divisão por zero)
    """
    norma_v1 = np.linalg.norm(vec1)
    norma_v2 = np.linalg.norm(vec2)

    # Proteção: evita divisão por zero
    if norma_v1 == 0 or norma_v2 == 0:
        return 0.0

    # Produto escalar: soma de v1[i] * v2[i]
    produto_escalar = np.dot(vec1, vec2)

    return float(produto_escalar / (norma_v1 * norma_v2))


# ============================================
# TESTE RÁPIDO: Vetores na mesma direção!
# ============================================
a = np.array([1, 0, 0])   # aponta para direita
b = np.array([2, 0, 0])   # também aponta para direita (só é mais longo)

sim = similaridade_cosseno(a, b)
print(f"Similaridade entre [1,0,0] e [2,0,0]: {sim:.4f}")
print("→ Mesma direção, similaridade = 1.0 (máxima!) ")


Similaridade entre [1,0,0] e [2,0,0]: 1.0000
→ Mesma direção, similaridade = 1.0 (máxima!) 


Distância de cosseno

In [27]:
def distancia_cosseno(vec1: np.ndarray, vec2: np.ndarray) -> float:
    """
    Calcula a Distância de Cosseno entre dois vetores.

    Fórmula: d = 1 - similaridade_cosseno(A, B)

    Retorna:
        float entre 0 e 2
        - 0.0 = idênticos
        - 1.0 = sem relação
        - 2.0 = completamente opostos

    É a métrica MAIS USADA para comparar embeddings de texto!
    """
    return float(1.0 - similaridade_cosseno(vec1, vec2))


print("✅ Funções de distância definidas com sucesso!")


✅ Funções de distância definidas com sucesso!


 # Testando com embeddings reais

In [28]:

modelo = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Modelo carregado!")

# Frases com contexto (não palavras soltas!O modelo não entende palavras soltas)
palavras_e_frases = {
    "gato":      "gato é um animal de estimação felino",
    "felino":    "felino é um animal mamífero da família dos gatos",
    "cachorro":  "cachorro é um animal de estimação canino",
    "carro":     "carro é um veículo automotor de quatro rodas",
    "caminhão":  "caminhão é um veículo de transporte de carga",
    "moto":      "moto é um veículo motorizado de duas rodas",
    "banana":    "banana é uma fruta tropical amarela",
    "maçã":      "maçã é uma fruta vermelha ou verde",
    "goiaba":    "goiaba é uma fruta tropical brasileira"
}

embeddings = {}
for palavra, frase in palavras_e_frases.items():
    embeddings[palavra] = modelo.encode(frase)

print(f"✅ Embeddings gerados para {len(palavras_e_frases)} palavras!")
print(f"   Cada embedding tem {len(embeddings['gato'])} dimensões")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Modelo carregado!
✅ Embeddings gerados para 9 palavras!
   Cada embedding tem 384 dimensões


Comparando pares de palavras

In [29]:
# ============================================
# TESTE 1: gato vs cachorro (animais)
# ============================================
print("=" * 60)
print(" gato  vs   cachorro")
print("=" * 60)

d_euc = distancia_euclidiana(embeddings["gato"], embeddings["cachorro"])
d_cos = distancia_cosseno(embeddings["gato"], embeddings["cachorro"])
sim   = similaridade_cosseno(embeddings["gato"], embeddings["cachorro"])

print(f"  Distância Euclidiana: {d_euc:.4f}  (quanto menor = mais parecido)")
print(f"  Distância de Cosseno:  {d_cos:.4f}  (0 = idêntico, 1 = diferente)")
print(f"  Similaridade Cosseno:  {sim:.4f}   (1 = idêntico, 0 = diferente)")
print(f"  → São parecidos! Ambos são animais de estimação ✅")

print()

# ============================================
# TESTE 2: gato vs carro (coisas diferentes)
# ============================================
print("=" * 60)
print(" gato  vs  carro")
print("=" * 60)

d_euc = distancia_euclidiana(embeddings["gato"], embeddings["carro"])
d_cos = distancia_cosseno(embeddings["gato"], embeddings["carro"])
sim   = similaridade_cosseno(embeddings["gato"], embeddings["carro"])

print(f"  Distância Euclidiana: {d_euc:.4f}")
print(f"  Distância de Cosseno:  {d_cos:.4f}")
print(f"  Similaridade Cosseno:  {sim:.4f}")
print(f"  → São bem diferentes! Nada a ver um com o outro ❌")

print()

# ============================================
# TESTE 3: gato vs felino (sinônimos!)
# ============================================
print("=" * 60)
print(" gato  vs   felino")
print("=" * 60)

d_euc = distancia_euclidiana(embeddings["gato"], embeddings["felino"])
d_cos = distancia_cosseno(embeddings["gato"], embeddings["felino"])
sim   = similaridade_cosseno(embeddings["gato"], embeddings["felino"])

print(f"  Distância Euclidiana: {d_euc:.4f}")
print(f"  Distância de Cosseno:  {d_cos:.4f}")
print(f"  Similaridade Cosseno:  {sim:.4f}")
print(f"  → Praticamente sinônimos! Maior similaridade de todas! 🎯")


 gato  vs   cachorro
  Distância Euclidiana: 0.8997  (quanto menor = mais parecido)
  Distância de Cosseno:  0.4047  (0 = idêntico, 1 = diferente)
  Similaridade Cosseno:  0.5953   (1 = idêntico, 0 = diferente)
  → São parecidos! Ambos são animais de estimação ✅

 gato  vs  carro
  Distância Euclidiana: 1.1309
  Distância de Cosseno:  0.6394
  Similaridade Cosseno:  0.3606
  → São bem diferentes! Nada a ver um com o outro ❌

 gato  vs   felino
  Distância Euclidiana: 0.7369
  Distância de Cosseno:  0.2715
  Similaridade Cosseno:  0.7285
  → Praticamente sinônimos! Maior similaridade de todas! 🎯


Buscar os .md anexados  (3 documentos da aula 2)

In [30]:

from pathlib import Path

# Os arquivos anexados ficam em /content/
CAMINHO_MARKDOWN = Path("/content")

print(f"📂 Procurando arquivos .md em: {CAMINHO_MARKDOWN}\n")

# Buscar todos os .md
arquivos_md = list(CAMINHO_MARKDOWN.glob("*.md"))

print(f"📄 {len(arquivos_md)} arquivo(s) encontrado(s):\n")
for arq in arquivos_md:
    print(f"   • {arq.name}")

if len(arquivos_md) == 0:
    print("\n⚠️  Nenhum .md encontrado! Anexe pelo ícone 📁 na barra lateral.")
else:
    print(f"\n✅ Pronto para continuar!")


📂 Procurando arquivos .md em: /content

📄 3 arquivo(s) encontrado(s):

   • escrita_academica_ia (3).md
   • bioetica_e_ia (3).md
   • twitter_algoritmo (3).md

✅ Pronto para continuar!


Carregar e dividir em linhas

In [31]:

def carregar_documentos_linhas(lista_arquivos):
    """Carrega arquivos .md e divide em linhas."""
    linhas = []
    metadados = []

    for arquivo in lista_arquivos:
        with open(arquivo, "r", encoding="utf-8") as f:
            conteudo = f.read()

        # Divide por quebras de linha e remove linhas vazias
        linhas_arquivo = [l.strip() for l in conteudo.split("\n") if l.strip()]

        for i, linha in enumerate(linhas_arquivo, start=1):
            linhas.append(linha)
            metadados.append({
                "arquivo": arquivo.name,
                "linha": i,
                "texto": linha
            })

    return linhas, metadados

# Executar
linhas, metadados = carregar_documentos_linhas(arquivos_md)

print(f"✅ Carregadas {len(linhas)} linhas de {len(arquivos_md)} arquivo(s)\n")
print("📋 Prévia (5 primeiras linhas):")
for m in metadados[:5]:
    print(f"   [{m['arquivo']}] linha {m['linha']}: {m['texto'][:80]}...")


✅ Carregadas 80 linhas de 3 arquivo(s)

📋 Prévia (5 primeiras linhas):
   [escrita_academica_ia (3).md] linha 1: ## Escrita acadêmica ética, responsável e humana com inteligência artificial...
   [escrita_academica_ia (3).md] linha 2: <!-- image -->...
   [escrita_academica_ia (3).md] linha 3: Rafael Cardoso Sampaio...
   [escrita_academica_ia (3).md] linha 4: ## I ✉...
   [escrita_academica_ia (3).md] linha 5: I Programa Pós-Graduação em Ciência Política, Universidade Federal do Paraná, Cu...


 Função de busca semântica

In [32]:
import numpy as np

def busca_semantica(consulta, trechos, metadados, top_k=5):
    """
    Busca os trechos mais similares à consulta por similaridade de cosseno.
    """
    # 1. Embedding da consulta
    emb_consulta = modelo.encode([consulta])[0]

    # 2. Embeddings dos trechos
    emb_trechos = modelo.encode(trechos)

    # 3. Similaridade de cosseno
    norma_consulta = np.linalg.norm(emb_consulta)
    normas_trechos = np.linalg.norm(emb_trechos, axis=1)
    produtos = np.dot(emb_trechos, emb_consulta)
    similaridades = produtos / (norma_consulta * normas_trechos)

    # 4. Ordenar (maior → menor)
    indices_ordenados = np.argsort(similaridades)[::-1]

    # 5. Montar resultados
    resultados = []
    for idx in indices_ordenados[:top_k]:
        resultados.append({
            "similaridade": similaridades[idx],
            "texto": trechos[idx],
            "metadado": metadados[idx]
        })

    return resultados

print("✅ Função busca_semantica pronta!")

✅ Função busca_semantica pronta!


Testar busca por LINHAS

In [33]:
# ============================================
# CÉLULA 12 — Testar busca semântica por LINHAS
# ============================================
consulta = "inteligência artificial na área da saúde"

print(f"🔍 Buscando por: \"{consulta}\"\n")
print("=" * 60)

resultados = busca_semantica(consulta, linhas, metadados, top_k=5)

for i, res in enumerate(resultados, 1):
    print(f"\n🏆 Resultado #{i} — Similaridade: {res['similaridade']:.4f}")
    print(f"   📄 Arquivo: {res['metadado']['arquivo']}")
    print(f"   📍 Linha:   {res['metadado']['linha']}")
    print(f"   💬 Texto:   {res['texto'][:120]}{'...' if len(res['texto']) > 120 else ''}")


🔍 Buscando por: "inteligência artificial na área da saúde"


🏆 Resultado #1 — Similaridade: 0.5390
   📄 Arquivo: escrita_academica_ia (3).md
   📍 Linha:   1
   💬 Texto:   ## Escrita acadêmica ética, responsável e humana com inteligência artificial

🏆 Resultado #2 — Similaridade: 0.5264
   📄 Arquivo: escrita_academica_ia (3).md
   📍 Linha:   6
   💬 Texto:   Palavras-chave: escrita acadêmica, integridade científica, Inteligência Artificial Generativa, autoria, revisão narrativ...

🏆 Resultado #3 — Similaridade: 0.4604
   📄 Arquivo: bioetica_e_ia (3).md
   📍 Linha:   3
   💬 Texto:   ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial

🏆 Resultado #4 — Similaridade: 0.4532
   📄 Arquivo: escrita_academica_ia (3).md
   📍 Linha:   18
   💬 Texto:   A integração da inteligência artificial generativa (IAG) na produção de conhecimento acadêmico, acelerada por ferramenta...

🏆 Resultado #5 — Similaridade: 0.4526
   📄 Arquivo: escrita_academica_ia (3).md
   

Testar busca por PARÁGRAFOS

In [34]:

import re

def carregar_documentos_paragrafos(lista_arquivos):
    """Carrega arquivos .md e divide por parágrafos."""
    paragrafos = []
    metadados = []

    for arquivo in lista_arquivos:
        with open(arquivo, "r", encoding="utf-8") as f:
            conteudo = f.read()

        # Divide por parágrafos (duas ou mais quebras de linha)
        paragrafos_arquivo = re.split(r'\n\s*\n', conteudo)
        paragrafos_arquivo = [p.strip() for p in paragrafos_arquivo if p.strip()]

        for i, paragrafo in enumerate(paragrafos_arquivo, start=1):
            paragrafos.append(paragrafo)
            metadados.append({
                "arquivo": arquivo.name,
                "paragrafo": i,
                "texto": paragrafo[:200]
            })

    return paragrafos, metadados

# Carregar parágrafos
paragrafos, metadados_parag = carregar_documentos_paragrafos(arquivos_md)
print(f"✅ Carregados {len(paragrafos)} parágrafos de {len(arquivos_md)} arquivo(s)\n")

# Buscar
consulta = "inteligência artificial na área da saúde"
print(f"🔍 Buscando por: \"{consulta}\"\n")
print("=" * 60)

resultados = busca_semantica(consulta, paragrafos, metadados_parag, top_k=3)

for i, res in enumerate(resultados, 1):
    print(f"\n🏆 Resultado #{i} — Similaridade: {res['similaridade']:.4f}")
    print(f"   📄 Arquivo:    {res['metadado']['arquivo']}")
    print(f"   📍 Parágrafo:  {res['metadado']['paragrafo']}")
    print(f"   💬 Texto:      {res['texto'][:150]}...")


✅ Carregados 80 parágrafos de 3 arquivo(s)

🔍 Buscando por: "inteligência artificial na área da saúde"


🏆 Resultado #1 — Similaridade: 0.5390
   📄 Arquivo:    escrita_academica_ia (3).md
   📍 Parágrafo:  1
   💬 Texto:      ## Escrita acadêmica ética, responsável e humana com inteligência artificial...

🏆 Resultado #2 — Similaridade: 0.5264
   📄 Arquivo:    escrita_academica_ia (3).md
   📍 Parágrafo:  6
   💬 Texto:      Palavras-chave: escrita acadêmica, integridade científica, Inteligência Artificial Generativa, autoria, revisão narrativa....

🏆 Resultado #3 — Similaridade: 0.4604
   📄 Arquivo:    bioetica_e_ia (3).md
   📍 Parágrafo:  3
   💬 Texto:      ## Entre o algoritmo e o Juramento de Hipócrates: bioética na era da inteligência artificial...
